# Test: Llama-4-Scout-17B-16E-Instruct — Validator V3

MoE (109B total, 17B active, 16 experts) — GPU 5, Port 8004

**Prerequisites:** vLLM server running on port 8004

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from models.utils import Llama4Scout

model = Llama4Scout()
print('Model config:')
model.get_config()

/home/student/.conda/envs/agenticcyops/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model config:


{'model_id': 'meta-llama/Llama-4-Scout-17B-16E-Instruct',
 'model_path': '/storage/data/AgenticCyOps_Private/models/meta-llama/Llama-4-Scout-17B-16E-Instruct',
 'role': 'validator_3',
 'architecture': 'MoE',
 'total_params': '109B',
 'active_params': '17B',
 'num_experts': 16,
 'gpu_assignment': '4,5',
 'port': 8004,
 'base_url': 'http://localhost:8004/v1',
 'tool_call_parser': 'llama4_pythonic'}

## 1. Health Check

In [2]:
assert model.health_check(), 'Server not running on port 8004!'
print('Health check passed')

Health check passed


## 2. List Models

In [3]:
models = model.list_models()
for m in models:
    print(f'  {m.id}')

  /storage/data/AgenticCyOps_Private/models/meta-llama/Llama-4-Scout-17B-16E-Instruct


## 3. Chat Completions

In [4]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

resp = model.chat(messages, max_tokens=100)
print('Basic chat:', resp.choices[0].message.content)

Basic chat: A lateral movement attack is a type of cyber attack where an attacker gains access to a network and then moves laterally across multiple systems, compromising additional accounts and systems to escalate privileges and gain unauthorized access to sensitive data.


In [5]:
# Deterministic
resp = model.chat_deterministic(messages, max_tokens=100)
print('Deterministic:', resp.choices[0].message.content)

Deterministic: A lateral movement attack is a type of cyber attack where an attacker gains access to a network and then moves laterally across multiple systems, compromising additional accounts and systems to escalate privileges and gain unauthorized access to sensitive data.


In [6]:
# Creative
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', resp.choices[0].message.content)

Creative: A lateral movement attack is a type of cyber attack where an attacker gains access to a network and then moves laterally across multiple systems, compromising additional accounts and systems to escalate privileges and gain further access.


In [7]:
# Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

Streaming: A lateral

 movement attack is a type of cyber attack where an attacker gains access to a network and then moves laterally across multiple systems, compromising additional accounts and systems to escalate privileges and gain unauthorized access to sensitive data.


## 4. Validate (Primary Use Case)

In [8]:
# Safe proposal
resp = model.validate(
    proposal='Isolate host WS-042 from the network due to confirmed lateral movement.',
    context='Alert: Lateral movement from WS-042 to DC-01 via PsExec. Source 10.0.5.42.'
)
print('Safe proposal:')
print(resp.choices[0].message.content)

Safe proposal:
{"approved": true, "confidence": 0.9, "reasoning": "The proposed action to isolate host WS-042 from the network is appropriate and necessary given the incident context of confirmed lateral movement via PsExec. This action will prevent further potential malicious activity from spreading and is a standard response to contain threats.", "risks": ["Potential business disruption if WS-042 is a critical host", "Possible false positive if the lateral movement was a legitimate administrative action"]}


In [9]:
# Dangerous proposal
resp = model.validate(
    proposal='Revoke all domain admin credentials immediately across all 500 accounts.',
    context='Alert: Single phishing email. No evidence of credential compromise.'
)
print('Dangerous proposal:')
print(resp.choices[0].message.content)

Dangerous proposal:
{"approved": false, 
"confidence": 0.2, 
"reasoning": "The proposed action of revoking all domain admin credentials across 500 accounts seems extreme and not directly related to the incident context provided. The incident is a single phishing email with no evidence of credential compromise. Such an action could lead to significant disruption of service and is not a proportionate response to the threat level presented.", 
"risks": ["Disruption of critical services", "Potential for accidental denial of service to legitimate users", "Increased workload for IT staff to reissue credentials and manage access recovery", "Possible negative impact on productivity and business operations"]}


In [10]:
# Batch validate
proposals = [
    {'proposal': 'Block IP 10.0.5.12 at firewall.', 'context': 'Confirmed C2 from 10.0.5.12.'},
    {'proposal': 'Delete all firewall rules.', 'context': 'Minor config drift detected.'},
]
results = model.batch_validate(proposals)
for i, r in enumerate(results):
    print(f'\nProposal {i}: {r.choices[0].message.content[:120]}...')


Proposal 0: {"approved": true, 
"confidence": 0.9, 
"reasoning": "The proposed action to block IP 10.0.5.12 at the firewall is a dir...

Proposal 1: {"approved": false, "confidence": 0.9, "reasoning": "Deleting all firewall rules could introduce significant security ri...


## 5. Tool Calling

In [11]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'query_siem',
        'description': 'Search SIEM logs',
        'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
    }
}]
tc_messages = [{'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12'}]

try:
    resp = model.tool_call(tc_messages, tools)
    tc = resp.choices[0].message.tool_calls
    if tc:
        print(f'Tool call: {tc[0].function.name}({tc[0].function.arguments})')
    else:
        print('No tool call (expected if served without --enable-auto-tool-choice)')
except Exception as e:
    print(f'Tool calling not available (expected for default validator config): {e}')

Tool call: query_siem({"query": "src_ip:10.0.5.12 AND event_type:failed_login"})


## 6. Structured Output

In [12]:
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Failed SSH from 10.0.5.12". Return {"severity": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

JSON mode: {"severity": "low", "confidence": 0.5}


## 7. Batch Chat

In [13]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.choices[0].message.content}')

Batch 0: Phishing is a type of cybercrime where attackers use fake emails, messages, or websites to trick individuals into revealing sensitive information, such as login credentials, financial information, or personal data.
Batch 1: Ransomware is a type of malicious software that encrypts a victim's files or locks their device and demands a ransom payment in exchange for the decryption key or unlock code.


## 8. Token Usage

In [14]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

Prompt tokens:     34
Completion tokens: 45
Total tokens:      79


## 9. Get Config

In [15]:
import json
print(json.dumps(model.get_config(), indent=2))

{
  "model_id": "meta-llama/Llama-4-Scout-17B-16E-Instruct",
  "model_path": "/storage/data/AgenticCyOps_Private/models/meta-llama/Llama-4-Scout-17B-16E-Instruct",
  "role": "validator_3",
  "architecture": "MoE",
  "total_params": "109B",
  "active_params": "17B",
  "num_experts": 16,
  "gpu_assignment": "4,5",
  "port": 8004,
  "base_url": "http://localhost:8004/v1",
  "tool_call_parser": "llama4_pythonic"
}


## Summary

All tests passed if no cells raised exceptions above.